# Week 5 Assignment
## Spark Fundamentals, Data Cleaning, Transformations and Aggregations

### Objective
Understand Spark fundamentals and perform data cleaning, transformation, filtering, grouping and aggregation using PySpark DataFrames.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5 Assignment") \
    .getOrCreate()

print("Spark Started Successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/22 19:41:26 WARN Utils: Your hostname, Himanshus-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.126 instead (on interface en0)
26/06/22 19:41:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 19:41:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Started Successfully!


## Creating Sample Dataset
Since no dataset was provided, a custom dataset is created for performing data cleaning and transformations.

In [2]:
data = [
    (1, "Alice", 25, "West", "Electronics", 500, "Premium", "alice@gmail.com"),
    (2, "Bob", 30, "East", "Clothing", 300, "Basic", "bob@gmail.com"),
    (3, "Charlie", 22, "West", "Electronics", None, "Premium", None),
    (4, "David", 28, "North", "Furniture", 700, "Premium", "david@gmail.com"),
    (5, "Eva", 19, "West", "Clothing", 200, "Premium", "eva@gmail.com"),
    (6, "", 27, "South", "Electronics", 450, "Basic", "john@gmail.com"),
    (1, "Alice", 25, "West", "Electronics", 500, "Premium", "alice@gmail.com")
]

columns = [
    "store_id",
    "username",
    "age",
    "region",
    "product_category",
    "price",
    "subscription",
    "email"
]

df = spark.createDataFrame(data, columns)

df.show()

+--------+--------+---+------+----------------+-----+------------+---------------+
|store_id|username|age|region|product_category|price|subscription|          email|
+--------+--------+---+------+----------------+-----+------------+---------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
|       2|     Bob| 30|  East|        Clothing|  300|       Basic|  bob@gmail.com|
|       3| Charlie| 22|  West|     Electronics| NULL|     Premium|           NULL|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|
|       6|        | 27| South|     Electronics|  450|       Basic| john@gmail.com|
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
+--------+--------+---+------+----------------+-----+------------+---------------+



# Q1. Limitations of MapReduce

### Answer

1. MapReduce performs disk-based processing which is slow.
2. Intermediate results are repeatedly written to disk.
3. Iterative machine learning algorithms become inefficient.
4. Batch processing only.
5. Programming is complex.

Spark is preferred because it performs in-memory processing, is faster and supports SQL, machine learning and streaming.

# Q2. In-Memory Computing

Spark stores intermediate results in RAM instead of writing them repeatedly to disk.

Therefore iterative algorithms run much faster than traditional MapReduce.

# Q3. Remove duplicates based on user_id and transaction_date

In [3]:
sample_data = [
(101,"2025-06-01",1000),
(101,"2025-06-01",1000),
(102,"2025-06-02",1500)
]

sample_columns = ["user_id","transaction_date","amount"]

df1 = spark.createDataFrame(sample_data,sample_columns)

df1.show()

df1 = df1.dropDuplicates(["user_id","transaction_date"])

df1.show()

+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|    101|      2025-06-01|  1000|
|    101|      2025-06-01|  1000|
|    102|      2025-06-02|  1500|
+-------+----------------+------+

+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|    101|      2025-06-01|  1000|
|    102|      2025-06-02|  1500|
+-------+----------------+------+



# Q4. Average sale amount for West region grouped by product category

In [4]:
from pyspark.sql.functions import avg

df.filter(df.region=="West") \
  .groupBy("product_category") \
  .agg(avg("price").alias("average_sale_amount")) \
  .show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|     Electronics|              500.0|
|        Clothing|              200.0|
+----------------+-------------------+



# Q5. Difference between na.drop() and na.fill()

na.drop():
Removes rows containing null values.

na.fill():
Replaces null values with specified values.

In [5]:
df.na.fill({"email":"Unknown"}).show()

+--------+--------+---+------+----------------+-----+------------+---------------+
|store_id|username|age|region|product_category|price|subscription|          email|
+--------+--------+---+------+----------------+-----+------------+---------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
|       2|     Bob| 30|  East|        Clothing|  300|       Basic|  bob@gmail.com|
|       3| Charlie| 22|  West|     Electronics| NULL|     Premium|        Unknown|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|
|       6|        | 27| South|     Electronics|  450|       Basic| john@gmail.com|
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
+--------+--------+---+------+----------------+-----+------------+---------------+



# Q6. Count records for each city where count >100

In [6]:
from pyspark.sql.functions import count

df.groupBy("region") \
  .agg(count("*").alias("total_count")) \
  .filter("total_count > 100") \
  .show()

+------+-----------+
|region|total_count|
+------+-----------+
+------+-----------+



# Q7. Immutability of Spark DataFrames

Spark DataFrames are immutable.

Operations like dropping columns or renaming columns do not modify the original DataFrame. Instead, a new DataFrame is created.

# Q8. Filter age between 18 and 30 and subscription = Premium

In [7]:
df.filter(
(df.age>=18) &
(df.age<=30) &
(df.subscription=="Premium")
).show()

+--------+--------+---+------+----------------+-----+------------+---------------+
|store_id|username|age|region|product_category|price|subscription|          email|
+--------+--------+---+------+----------------+-----+------------+---------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
|       3| Charlie| 22|  West|     Electronics| NULL|     Premium|           NULL|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
+--------+--------+---+------+----------------+-----+------------+---------------+



# Q9. Why handle null values before aggregation?

Null values may lead to inaccurate calculations.

Therefore null values should be removed or replaced before using functions such as sum() and avg().

# Q10. Convert raw_timestamp to TimestampType and rename it to event_time

In [8]:
from pyspark.sql.functions import current_timestamp

df2 = df.withColumn("raw_timestamp",current_timestamp())

df2 = df2.withColumnRenamed("raw_timestamp","event_time")

df2.show()

+--------+--------+---+------+----------------+-----+------------+---------------+--------------------+
|store_id|username|age|region|product_category|price|subscription|          email|          event_time|
+--------+--------+---+------+----------------+-----+------------+---------------+--------------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|2026-06-22 19:50:...|
|       2|     Bob| 30|  East|        Clothing|  300|       Basic|  bob@gmail.com|2026-06-22 19:50:...|
|       3| Charlie| 22|  West|     Electronics| NULL|     Premium|           NULL|2026-06-22 19:50:...|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|2026-06-22 19:50:...|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|2026-06-22 19:50:...|
|       6|        | 27| South|     Electronics|  450|       Basic| john@gmail.com|2026-06-22 19:50:...|
|       1|   Alice| 25|  West|     Electronics|  500|     Premiu

# Q11. Shuffle Operation

Shuffle refers to moving data across partitions.

Operations like groupBy() require records having the same key to come together.

Therefore groupBy() is called a wide transformation.

# Q12. Remove rows where email is null or username is empty

In [9]:
df.filter(
(df.email.isNotNull()) &
(df.username != "")
).show()

+--------+--------+---+------+----------------+-----+------------+---------------+
|store_id|username|age|region|product_category|price|subscription|          email|
+--------+--------+---+------+----------------+-----+------------+---------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
|       2|     Bob| 30|  East|        Clothing|  300|       Basic|  bob@gmail.com|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
+--------+--------+---+------+----------------+-----+------------+---------------+



# Q13. Multiple aggregations using agg()

In [10]:
from pyspark.sql.functions import min,max,mean

df.agg(
min("price").alias("Minimum Price"),
max("price").alias("Maximum Price"),
mean("price").alias("Average Price")
).show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|          200|          700|441.6666666666667|
+-------------+-------------+-----------------+



# Q14. Risk of inferSchema=True

When data contains inconsistent date formats, inferSchema=True may incorrectly identify data types.

This can lead to errors and incorrect analysis.

# Q15. Final Processing Pipeline

1. Remove duplicates.
2. Fill null prices with 0.
3. Group by store_id.
4. Calculate total revenue.

In [11]:
from pyspark.sql.functions import sum

result = (
df.dropDuplicates()
.na.fill({"price":0})
.groupBy("store_id")
.agg(sum("price").alias("total_revenue"))
)

result.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       6|          450|
|       5|          200|
|       1|          500|
|       3|            0|
|       2|          300|
|       4|          700|
+--------+-------------+



In [12]:
print(df.columns)

['store_id', 'username', 'age', 'region', 'product_category', 'price', 'subscription', 'email']


In [13]:
df.printSchema()

root
 |-- store_id: long (nullable = true)
 |-- username: string (nullable = true)
 |-- age: long (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)



#Drop rows containing null values

In [14]:
df.na.drop().show()

+--------+--------+---+------+----------------+-----+------------+---------------+
|store_id|username|age|region|product_category|price|subscription|          email|
+--------+--------+---+------+----------------+-----+------------+---------------+
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
|       2|     Bob| 30|  East|        Clothing|  300|       Basic|  bob@gmail.com|
|       4|   David| 28| North|       Furniture|  700|     Premium|david@gmail.com|
|       5|     Eva| 19|  West|        Clothing|  200|     Premium|  eva@gmail.com|
|       6|        | 27| South|     Electronics|  450|       Basic| john@gmail.com|
|       1|   Alice| 25|  West|     Electronics|  500|     Premium|alice@gmail.com|
+--------+--------+---+------+----------------+-----+------------+---------------+



# Check inconsistent data

In [15]:
df.filter(df.username == "").show()

+--------+--------+---+------+----------------+-----+------------+--------------+
|store_id|username|age|region|product_category|price|subscription|         email|
+--------+--------+---+------+----------------+-----+------------+--------------+
|       6|        | 27| South|     Electronics|  450|       Basic|john@gmail.com|
+--------+--------+---+------+----------------+-----+------------+--------------+



## Change datatype

In [16]:
from pyspark.sql.functions import col

df = df.withColumn("age", col("age").cast("integer"))

df.printSchema()

root
 |-- store_id: long (nullable = true)
 |-- username: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)



# Observations

- Duplicate records were successfully removed.
- Missing values were handled using fill() and drop().
- Data was filtered using age, category, and region.
- Aggregation functions such as count(), sum(), avg(), min(), and max() were applied.
- GroupBy operations involved shuffle and wide transformations.
- A complete data processing pipeline was implemented using PySpark DataFrames.

In [17]:
df.toPandas().to_csv("dataset.csv", index=False)